In [1]:
import json
import re
import codecs
import urllib.request
import requests 
from bs4 import BeautifulSoup # for HTML parsing
import time # for sleep
from tqdm import tqdm # for progress tracker
import urllib3 # to disable SSL warnings 

# Disable the SSL warnings
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [2]:
# Opens and reads the JSON file gained from JT's data vis scraping tool 
with open('articles-and-dvcs.json', 'r') as f:
    results = json.load(f)
    
results.reverse()

In [3]:
pages = []

In [4]:
for article in tqdm(results, desc="Processing articles"):

    #get the article theme from the url
    split = article['doc_uri'].split('/')
    article['theme'] = split[3]

    if article['vis_urls']:
        article['vis'] = []
        for i, vis in enumerate(tqdm(article['vis_urls'], desc="Fetching visualizations", leave=False)):
            if 'datadownload.xlsx' in vis:
                continue
            url = "https://www.ons.gov.uk" + vis
            html = requests.get(url, verify=False).text

            # Only sleep between requests, not after the last one
            # if i < len(article['vis_urls']) - 1: 
            #     time.sleep(0.5)
            
            soup = BeautifulSoup(html, "lxml")
    
# Most common: <meta name="template" content="...">
            meta = soup.find("meta", attrs={"name": "template"})
            template = meta.get("content") if meta else None
            article["vis"].append({"url": vis, "chart_type": template})

# Fallback: any meta where name/property contains "template"
            if template is None:
                meta = soup.find("meta", attrs={"name": lambda v: v and "template" in v.lower()}) \
                    or soup.find("meta", attrs={"property": lambda v: v and "template" in v.lower()})
                template = meta.get("content") if meta else None
            # print(template)

            # if this release is not currently in the list pages add it
        if not pages:
                pages.append(article)

            # if there is a previous version of this article replace it with the newer version if it has data vis in it
        elif article['vis_urls']:
            article_titles = list(map(lambda article: article['title'], pages))
            if article['title'] in article_titles:
                for index, item in enumerate(pages):
                    if item['title'] == article['title']:
                        item = article
                        break
                    else:
                        index = -1
            else:
                pages.append(article)

Processing articles:  13%|█▎        | 223/1674 [02:06<05:43,  4.23it/s]  C:\Users\rickas\AppData\Local\Temp\ipykernel_24064\4030569374.py:19: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html, "lxml")
Processing articles: 100%|██████████| 1674/1674 [12:08<00:00,  2.30it/s]


In [5]:
# sort list by release date from newest to oldest
pages.sort(key=lambda x: x['release_date'], reverse=True)

In [6]:
json_str = json.dumps(pages, indent=6)
with open("data.json", "w") as f:
    f.write(json_str)